# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook demonstrates loading, exploring, and processing the [FAIR^2 mass spectrometry](https://sen.science/doi/10.71728/senscience.vq4a-28xa) dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a [Croissant schema](https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json) URL.

In [ ]:
# Install mlcroissant if not already installed
!pip install --quiet mlcroissant

## 1. Data Loading
Load the dataset metadata and preview its description using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset Croissant schema URL
croissant_url = "https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json"

# Load the dataset
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview
Review available record sets, their fields, and all entity `@id` values.

We enumerate all record sets present, showing their `@id`, names, and the associated fields (by `@id`):

In [ ]:
# Display all record sets and their fields by @id
record_sets = dataset.record_sets
for rs in record_sets:
    print(f"RecordSet @id: {rs.id}")
    print(f"  Name: {rs.name}")
    print(f"  Description: {rs.description}")
    fields = getattr(rs, 'fields', [])
    print(f"  Fields (@id):")
    for field in fields:
        print(f"    • {field.id} (name={field.name}, type={field.data_type})")
    print('-' * 60)

## 3. Data Extraction
We load the data from a chosen record set (using its `@id`) into a DataFrame for analysis.

***Note:*** You must refer to record sets and fields by their `@id` exactly as above.

In [ ]:
# List all record set @ids in the dataset
record_set_ids = [rs.id for rs in dataset.record_sets]
print(f"Available record sets (@id): {record_set_ids}")

# We'll load the primary record set for this dataset, which typically contains the main data results (you may change below)
main_record_set_id = record_set_ids[0]

# Load records for each record set into pandas DataFrames
dataframes = {}
for rs_id in record_set_ids:
    records = list(dataset.records(record_set=rs_id))
    df = pd.DataFrame(records)
    dataframes[rs_id] = df

print(f"Loaded columns for '{main_record_set_id}':")
print(dataframes[main_record_set_id].columns.tolist())

# Display a sample of the main data
dataframes[main_record_set_id].head()

## 4. Exploratory Data Analysis (EDA)
Let's process the core protein abundance table. We'll select a numeric field (by its `@id` as found above), filter, normalize, and analyze.

### Steps
- Select a numeric field (e.g., a peptide count or abundance value)
- Filter all rows with field value > threshold (e.g., top peptides)
- Normalize that field
- Optionally group by a categorical field (e.g., protein or sample)

In [ ]:
# Substitute these with a numeric and a group field's @id from the overview
df = dataframes[main_record_set_id]

# You can adjust these fields (@id) if needed:
numeric_field_id = None
group_field_id = None

# Try to auto-select a likely numeric field by name
for col in df.columns:
    if "count" in col.lower() or "abundance" in col.lower() or "int" in str(df[col].dtype):
        numeric_field_id = col
        break
if numeric_field_id is None:
    # Just use the first field if cannot auto-detect
    numeric_field_id = df.columns[0]

# Try to auto-select a possible grouping field
for col in df.columns:
    if (col != numeric_field_id) and ("protein" in col.lower() or "sample" in col.lower() or "access" in col.lower()):
        group_field_id = col
        break
# If not found, leave as None

# Display the selection
print(f"Numeric field selected (@id): {numeric_field_id}")
if group_field_id:
    print(f"Grouping by (@id): {group_field_id}")

# Attempt to convert numeric field to float
df[numeric_field_id] = pd.to_numeric(df[numeric_field_id], errors='coerce')

threshold = df[numeric_field_id].quantile(0.9)  # Use 90th percentile as threshold for demonstration
filtered_df = df[df[numeric_field_id] > threshold].copy()
print(f"Filtered records with {numeric_field_id} > {threshold:.2f} (top 10%):")
print(filtered_df.head())

filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
print(f"Normalized {numeric_field_id} for filtered records:")
print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

# Optional grouping
if group_field_id and group_field_id in filtered_df.columns:
    grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
    print(f"Grouped mean {numeric_field_id} by {group_field_id}:")
    print(grouped_df.head())

## 5. Visualization
We visualize the distribution of the selected numeric field and the effect of normalization. If a group field is present, we also plot aggregates.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

plt.figure(figsize=(8, 4))
sns.histplot(df[numeric_field_id].dropna(), kde=True, bins=30)
plt.title(f"Distribution of {numeric_field_id}")
plt.xlabel(numeric_field_id)
plt.ylabel("Count")
plt.show()

if f"{numeric_field_id}_normalized" in filtered_df.columns:
    plt.figure(figsize=(8, 4))
    sns.histplot(filtered_df[f"{numeric_field_id}_normalized"].dropna(), kde=True, bins=20)
    plt.title(f"Normalized {numeric_field_id} (Top Records)")
    plt.xlabel(f"{numeric_field_id}_normalized")
    plt.ylabel("Count")
    plt.show()

# If there's grouping, plot top groups
if group_field_id and group_field_id in df.columns:
    top_groups = df[group_field_id].value_counts().head(10).index
    plt.figure(figsize=(10, 4))
    sns.boxplot(x=group_field_id, y=numeric_field_id, data=df[df[group_field_id].isin(top_groups)])
    plt.title(f"{numeric_field_id} by {group_field_id} (Top 10 Groups)")
    plt.xticks(rotation=45)
    plt.show()

## 6. Conclusion
- We successfully loaded the FAIR^2 mass spectrometry dataset using `mlcroissant` with record sets and fields referenced by `@id`.
- Numeric field distributions were visualized and normalized; group-wise differences explored where possible.
- This workflow can be extended for deeper domain-specific analyses or ML tasks.

*Be sure to reference field and record set IDs using their exact `@id` for any downstream analysis or processing using Croissant-powered tools!*